# Nine-Ticker Session-Aligned Panel Builder

This notebook builds a 9-ticker session-aligned raw panel for:
`AAPL`, `AMD`, `AMZN`, `GOOGL`, `META`, `MSFT`, `NFLX`, `NVDA`, `TSLA`.

Scope:
- `AVGO` is excluded
- raw files in `data/equity_data/` are treated as read-only inputs
- `TSLA` uses the generic raw filenames
- the final panel keeps original source columns and missing values
- no additional feature engineering is applied in the final panel
- Reddit daily files are merged with their original values
- Google Trends and GDELT are minimally session-aligned onto trading dates

Expected outputs:
- `data/datasets/stock_panel_nine_tickers_session_aligned_raw.csv`
- `data/datasets/stock_panel_nine_tickers_session_aligned_summary.csv`
- `data/datasets/stock_panel_nine_tickers_session_aligned_metadata.json`
- audit CSVs for raw source selection and Reddit coverage

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)

In [2]:
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data' / 'equity_data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / 'data' / 'equity_data'
DATASETS_DIR = PROJECT_ROOT / 'data' / 'datasets'
DATASETS_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_DATASET_PATH = DATASETS_DIR / 'stock_panel_nine_tickers_session_aligned_raw.csv'
OUTPUT_SUMMARY_PATH = DATASETS_DIR / 'stock_panel_nine_tickers_session_aligned_summary.csv'
OUTPUT_METADATA_PATH = DATASETS_DIR / 'stock_panel_nine_tickers_session_aligned_metadata.json'
OUTPUT_SOURCE_AUDIT_PATH = DATASETS_DIR / 'stock_panel_nine_tickers_session_aligned_source_audit.csv'
OUTPUT_REDDIT_COVERAGE_PATH = DATASETS_DIR / 'stock_panel_nine_tickers_session_aligned_reddit_coverage_audit.csv'

ALIGNMENT_POLICY = 'calendar-day alternative data mapped to same or next trading session'
MIN_SUBM_POST_COVERAGE = 0.35
MIN_COMM_POST_COVERAGE = 0.55

COMPANY_NAMES = {
    'AAPL': 'Apple',
    'AMD': 'Advanced Micro Devices',
    'AMZN': 'Amazon',
    'GOOGL': 'Alphabet',
    'META': 'Meta',
    'MSFT': 'Microsoft',
    'NFLX': 'Netflix',
    'NVDA': 'NVIDIA',
    'TSLA': 'Tesla',
}

TICKER_SPECS = [
    {
        'ticker': 'AAPL',
        'company_name': COMPANY_NAMES['AAPL'],
        'stock_candidates': ['stock-prices-data_aapl.csv'],
        'google_candidates': ['google_trends_data_aapl.csv'],
        'reddit_subm_candidates': ['stock-reddit-data_aapl.csv'],
        'reddit_comm_candidates': ['stock-reddit-comments-data_aapl.csv'],
        'gdelt_candidates': ['gdelt_data_aapl.csv'],
    },
    {
        'ticker': 'AMD',
        'company_name': COMPANY_NAMES['AMD'],
        'stock_candidates': ['stock-prices-data_amd.csv'],
        'google_candidates': ['google_trends_data_amd.csv'],
        'reddit_subm_candidates': ['stock-reddit-data_amd.csv'],
        'reddit_comm_candidates': ['stock-reddit-comments-data_amd.csv'],
        'gdelt_candidates': ['gdelt_data_amd.csv'],
    },
    {
        'ticker': 'AMZN',
        'company_name': COMPANY_NAMES['AMZN'],
        'stock_candidates': ['stock-prices-data_amzn.csv'],
        'google_candidates': ['google_trends_data_amzn.csv'],
        'reddit_subm_candidates': ['stock-reddit-data_amzn.csv'],
        'reddit_comm_candidates': ['stock-reddit-comments-data_amzn.csv'],
        'gdelt_candidates': ['gdelt_data_amzn.csv'],
    },
    {
        'ticker': 'GOOGL',
        'company_name': COMPANY_NAMES['GOOGL'],
        'stock_candidates': ['stock-prices-data_googl.csv'],
        'google_candidates': ['google_trends_data_googl.csv'],
        'reddit_subm_candidates': ['stock-reddit-data_googl.csv'],
        'reddit_comm_candidates': ['stock-reddit-comments-data_googl.csv'],
        'gdelt_candidates': ['gdelt_data_googl.csv'],
    },
    {
        'ticker': 'META',
        'company_name': COMPANY_NAMES['META'],
        'stock_candidates': ['stock-prices-data_meta.csv'],
        'google_candidates': ['google_trends_data_meta.csv'],
        'reddit_subm_candidates': ['stock-reddit-data_meta.csv'],
        'reddit_comm_candidates': ['stock-reddit-comments-data_meta.csv'],
        'gdelt_candidates': ['gdelt_data_meta.csv'],
    },
    {
        'ticker': 'MSFT',
        'company_name': COMPANY_NAMES['MSFT'],
        'stock_candidates': ['stock-prices-data_msft.csv'],
        'google_candidates': ['google_trends_data_msft.csv'],
        'reddit_subm_candidates': ['stock-reddit-data_msft.csv'],
        'reddit_comm_candidates': ['stock-reddit-comments-data_msft.csv'],
        'gdelt_candidates': ['gdelt_data_msft.csv'],
    },
    {
        'ticker': 'NFLX',
        'company_name': COMPANY_NAMES['NFLX'],
        'stock_candidates': ['stock-prices-data_nflx.csv'],
        'google_candidates': ['google_trends_data_nflx.csv'],
        'reddit_subm_candidates': ['stock-reddit-data_nflx.csv'],
        'reddit_comm_candidates': ['stock-reddit-comments-data_nflx.csv'],
        'gdelt_candidates': ['gdelt_data_nflx.csv'],
    },
    {
        'ticker': 'NVDA',
        'company_name': COMPANY_NAMES['NVDA'],
        'stock_candidates': ['stock-prices-data_nvda.csv'],
        'google_candidates': ['google_trends_data_nvda.csv'],
        'reddit_subm_candidates': ['stock-reddit-data_nvda.csv'],
        'reddit_comm_candidates': ['stock-reddit-comments-data_nvda.csv'],
        'gdelt_candidates': ['gdelt_data_nvda.csv'],
    },
    {
        'ticker': 'TSLA',
        'company_name': COMPANY_NAMES['TSLA'],
        'stock_candidates': ['stock-prices-data.csv', 'stock-prices-data_tsla.csv'],
        'google_candidates': ['google_trends_data.csv', 'google_trends_data_tsla.csv'],
        'reddit_subm_candidates': ['stock-reddit-data.csv', 'stock-reddit-data_tsla.csv'],
        'reddit_comm_candidates': ['stock-reddit-comments-data.csv', 'stock-reddit-comments-data_tsla.csv'],
        'gdelt_candidates': ['gdelt_data.csv', 'gdelt_data_tsla.csv'],
    },
]

In [3]:
REDDIT_BASE_COLUMNS = [
    'reddit_posts',
    'reddit_weight_sum',
    'reddit_score_sum',
    'reddit_comments_sum',
    'reddit_vader_mean',
    'reddit_vader_sum',
    'reddit_vader_std',
    'reddit_finbert_mean',
    'reddit_finbert_sum',
    'reddit_finbert_std',
    'reddit_vader_weighted_mean',
    'reddit_finbert_weighted_mean',
    'reddit_sent_mean',
    'reddit_sent_sum',
    'reddit_sent_std',
]

GDELT_NUMERIC_COLUMNS = [
    'gdelt_articles',
    'gdelt_robust',
    'gdelt_sentiment_score',
]


def first_existing_path(candidates: list[str]) -> Path | None:
    for candidate in candidates:
        path = RAW_DIR / candidate
        if path.exists():
            return path
    return None


def read_csv_with_standard_date(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path)
    date_column = next((col for col in ('date', 'Date') if col in frame.columns), None)
    if date_column is None:
        raise KeyError(f'No supported date column found in {path}')
    frame = frame.rename(columns={date_column: 'date'})
    frame['date'] = pd.to_datetime(frame['date'], errors='coerce').dt.normalize()
    frame = frame.dropna(subset=['date']).copy()
    return frame


def coerce_numeric_columns(frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    working = frame.copy()
    for column in columns:
        if column not in working.columns:
            working[column] = np.nan
        working[column] = pd.to_numeric(working[column], errors='coerce')
    return working


def load_stock_frame(spec: dict) -> tuple[pd.DataFrame, Path]:
    path = first_existing_path(spec['stock_candidates'])
    if path is None:
        raise FileNotFoundError(f"No stock source found for {spec['ticker']}")
    frame = read_csv_with_standard_date(path)
    frame = coerce_numeric_columns(frame, ['stock_price', 'stock_volume'])
    frame = frame[['date', 'stock_price', 'stock_volume']].sort_values('date').reset_index(drop=True)
    return frame, path


def load_google_frame(spec: dict) -> tuple[pd.DataFrame, Path]:
    path = first_existing_path(spec['google_candidates'])
    if path is None:
        raise FileNotFoundError(f"No Google Trends source found for {spec['ticker']}")
    frame = read_csv_with_standard_date(path)
    frame = frame.rename(columns={'trends_score': 'google_trends_score'})
    frame = coerce_numeric_columns(frame, ['google_trends_score'])
    frame = frame[['date', 'google_trends_score']].sort_values('date').reset_index(drop=True)
    return frame, path


def load_gdelt_frame(spec: dict) -> tuple[pd.DataFrame, Path]:
    path = first_existing_path(spec['gdelt_candidates'])
    if path is None:
        raise FileNotFoundError(f"No GDELT source found for {spec['ticker']}")
    frame = read_csv_with_standard_date(path)
    frame = frame.rename(columns={'sentiment_score': 'gdelt_sentiment_score'})
    frame = coerce_numeric_columns(frame, GDELT_NUMERIC_COLUMNS)
    frame = frame[['date', 'gdelt_articles', 'gdelt_robust', 'gdelt_sentiment_score']]
    frame = frame.groupby('date', as_index=False).mean().sort_values('date').reset_index(drop=True)
    return frame, path


def load_reddit_daily_frame(spec: dict, source_key: str) -> tuple[pd.DataFrame, Path]:
    path = first_existing_path(spec[source_key])
    if path is None:
        raise FileNotFoundError(f"No Reddit source found for {spec['ticker']} and key {source_key}")
    frame = read_csv_with_standard_date(path)
    frame = coerce_numeric_columns(frame, REDDIT_BASE_COLUMNS)
    frame = frame[['date'] + REDDIT_BASE_COLUMNS].sort_values('date').reset_index(drop=True)
    return frame, path


def inspect_source(spec: dict, source_name: str, candidates: list[str]) -> dict:
    path = first_existing_path(candidates)
    if path is None:
        return {
            'ticker': spec['ticker'],
            'company_name': spec['company_name'],
            'source_name': source_name,
            'selected_file': None,
            'used_fallback': False,
            'rows': 0,
            'date_min': None,
            'date_max': None,
        }

    frame = read_csv_with_standard_date(path)
    return {
        'ticker': spec['ticker'],
        'company_name': spec['company_name'],
        'source_name': source_name,
        'selected_file': path.name,
        'used_fallback': path.name != candidates[0],
        'rows': int(len(frame)),
        'date_min': frame['date'].min().date().isoformat() if not frame.empty else None,
        'date_max': frame['date'].max().date().isoformat() if not frame.empty else None,
    }


def build_raw_source_audit(ticker_specs: list[dict]) -> pd.DataFrame:
    records = []
    source_map = {
        'stock': 'stock_candidates',
        'google_trends': 'google_candidates',
        'reddit_submissions': 'reddit_subm_candidates',
        'reddit_comments': 'reddit_comm_candidates',
        'gdelt': 'gdelt_candidates',
    }
    for spec in ticker_specs:
        for source_name, candidates_key in source_map.items():
            records.append(inspect_source(spec, source_name, spec[candidates_key]))
    return pd.DataFrame(records).sort_values(['ticker', 'source_name']).reset_index(drop=True)


def build_reddit_coverage_audit(ticker_specs: list[dict]) -> pd.DataFrame:
    records = []
    for spec in ticker_specs:
        subm, subm_path = load_reddit_daily_frame(spec, 'reddit_subm_candidates')
        comm, comm_path = load_reddit_daily_frame(spec, 'reddit_comm_candidates')
        records.append(
            {
                'ticker': spec['ticker'],
                'company_name': spec['company_name'],
                'subm_source_file': subm_path.name,
                'comm_source_file': comm_path.name,
                'subm_days': int(len(subm)),
                'subm_days_with_posts': int((subm['reddit_posts'] > 0).sum()),
                'subm_post_coverage': float((subm['reddit_posts'] > 0).mean()),
                'subm_finbert_null_days': int(subm['reddit_finbert_mean'].isna().sum()),
                'subm_finbert_zero_days': int(subm['reddit_finbert_mean'].fillna(np.nan).eq(0).sum()),
                'comm_days': int(len(comm)),
                'comm_days_with_posts': int((comm['reddit_posts'] > 0).sum()),
                'comm_post_coverage': float((comm['reddit_posts'] > 0).mean()),
                'comm_finbert_null_days': int(comm['reddit_finbert_mean'].isna().sum()),
                'comm_finbert_zero_days': int(comm['reddit_finbert_mean'].fillna(np.nan).eq(0).sum()),
            }
        )
    return pd.DataFrame(records).sort_values('ticker').reset_index(drop=True)

In [4]:
def build_session_calendar(trading_dates: pd.Series) -> pd.DataFrame:
    return (
        pd.Series(pd.to_datetime(trading_dates).dropna().sort_values().unique(), name='session_date')
        .to_frame()
        .reset_index(drop=True)
    )


def map_calendar_to_next_session(frame: pd.DataFrame, trading_dates: pd.Series) -> pd.DataFrame:
    if frame.empty:
        working = frame.copy()
        working['session_date'] = pd.NaT
        return working

    session_calendar = build_session_calendar(trading_dates)
    working = frame.sort_values('date').copy()
    mapped = pd.merge_asof(
        working,
        session_calendar,
        left_on='date',
        right_on='session_date',
        direction='forward',
        allow_exact_matches=True,
    )
    return mapped.dropna(subset=['session_date']).copy()


def align_google_to_sessions(google_df: pd.DataFrame, trading_dates: pd.Series) -> pd.DataFrame:
    mapped = map_calendar_to_next_session(google_df, trading_dates)
    aligned = mapped.groupby('session_date', as_index=False).agg(
        google_trends_score=('google_trends_score', 'mean')
    )
    return aligned.rename(columns={'session_date': 'date'})


def align_gdelt_to_sessions(gdelt_df: pd.DataFrame, trading_dates: pd.Series) -> pd.DataFrame:
    mapped = map_calendar_to_next_session(gdelt_df, trading_dates)
    aligned = mapped.groupby('session_date', as_index=False).agg(
        gdelt_articles=('gdelt_articles', 'sum'),
        gdelt_robust=('gdelt_robust', 'mean'),
        gdelt_sentiment_score=('gdelt_sentiment_score', 'mean'),
    )
    return aligned.rename(columns={'session_date': 'date'})


def prepare_reddit_for_merge(reddit_df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    rename_map = {column: f'{prefix}_{column}' for column in REDDIT_BASE_COLUMNS}
    return reddit_df.rename(columns=rename_map)

In [5]:
def build_panel_for_ticker(spec: dict) -> pd.DataFrame:
    stock_df, _ = load_stock_frame(spec)
    google_df, _ = load_google_frame(spec)
    gdelt_df, _ = load_gdelt_frame(spec)
    subm_df, _ = load_reddit_daily_frame(spec, 'reddit_subm_candidates')
    comm_df, _ = load_reddit_daily_frame(spec, 'reddit_comm_candidates')

    panel = stock_df.copy()
    panel['ticker'] = spec['ticker']
    panel['company_name'] = spec['company_name']

    panel = panel.merge(
        prepare_reddit_for_merge(subm_df, 'subm'),
        on='date',
        how='left',
    )
    panel = panel.merge(
        prepare_reddit_for_merge(comm_df, 'comm'),
        on='date',
        how='left',
    )
    panel = panel.merge(
        align_google_to_sessions(google_df, stock_df['date']),
        on='date',
        how='left',
    )
    panel = panel.merge(
        align_gdelt_to_sessions(gdelt_df, stock_df['date']),
        on='date',
        how='left',
    )

    ordered_columns = [
        'date',
        'ticker',
        'company_name',
        'stock_price',
        'stock_volume',
        'subm_reddit_posts',
        'subm_reddit_weight_sum',
        'subm_reddit_score_sum',
        'subm_reddit_comments_sum',
        'subm_reddit_vader_mean',
        'subm_reddit_vader_sum',
        'subm_reddit_vader_std',
        'subm_reddit_finbert_mean',
        'subm_reddit_finbert_sum',
        'subm_reddit_finbert_std',
        'subm_reddit_vader_weighted_mean',
        'subm_reddit_finbert_weighted_mean',
        'subm_reddit_sent_mean',
        'subm_reddit_sent_sum',
        'subm_reddit_sent_std',
        'comm_reddit_posts',
        'comm_reddit_weight_sum',
        'comm_reddit_score_sum',
        'comm_reddit_comments_sum',
        'comm_reddit_vader_mean',
        'comm_reddit_vader_sum',
        'comm_reddit_vader_std',
        'comm_reddit_finbert_mean',
        'comm_reddit_finbert_sum',
        'comm_reddit_finbert_std',
        'comm_reddit_vader_weighted_mean',
        'comm_reddit_finbert_weighted_mean',
        'comm_reddit_sent_mean',
        'comm_reddit_sent_sum',
        'comm_reddit_sent_std',
        'google_trends_score',
        'gdelt_articles',
        'gdelt_robust',
        'gdelt_sentiment_score',
    ]
    return panel[ordered_columns].sort_values('date').reset_index(drop=True)


def build_panel(ticker_specs: list[dict]) -> pd.DataFrame:
    frames = [build_panel_for_ticker(spec) for spec in ticker_specs]
    return pd.concat(frames, ignore_index=True).sort_values(['date', 'ticker']).reset_index(drop=True)


def build_panel_summary(panel: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {'metric': 'rows', 'value': int(len(panel))},
            {'metric': 'tickers', 'value': int(panel['ticker'].nunique())},
            {'metric': 'date_min', 'value': panel['date'].min().date().isoformat()},
            {'metric': 'date_max', 'value': panel['date'].max().date().isoformat()},
            {'metric': 'subm_finbert_null_rate', 'value': float(panel['subm_reddit_finbert_mean'].isna().mean())},
            {'metric': 'comm_finbert_null_rate', 'value': float(panel['comm_reddit_finbert_mean'].isna().mean())},
        ]
    )


def build_metadata(source_audit: pd.DataFrame, reddit_coverage_audit: pd.DataFrame, panel: pd.DataFrame) -> dict:
    return {
        'alignment_policy': ALIGNMENT_POLICY,
        'rows': int(len(panel)),
        'tickers_included': sorted(panel['ticker'].unique().tolist()),
        'output_path': str(OUTPUT_DATASET_PATH),
        'summary_path': str(OUTPUT_SUMMARY_PATH),
        'source_audit_path': str(OUTPUT_SOURCE_AUDIT_PATH),
        'reddit_coverage_path': str(OUTPUT_REDDIT_COVERAGE_PATH),
        'notes': [
            'Raw source files are read-only inputs.',
            'TSLA uses the generic raw filenames as the primary source mapping.',
            'Reddit daily files are merged with original source values.',
            'Google Trends is minimally mapped from calendar days to the next trading session using a mean.',
            'GDELT is minimally mapped from calendar days to the next trading session using sum for gdelt_articles and mean for gdelt_robust and gdelt_sentiment_score.',
            'No additional missingness flags or coverage features are added to the final panel.',
            'AVGO is intentionally excluded from this panel.',
        ],
        'raw_source_rows': source_audit.to_dict(orient='records'),
        'coverage_rows': reddit_coverage_audit.to_dict(orient='records'),
    }

In [6]:
source_audit = build_raw_source_audit(TICKER_SPECS)
reddit_coverage_audit = build_reddit_coverage_audit(TICKER_SPECS)

print('Raw source audit:')
print(source_audit.to_string(index=False))
print()
print('Reddit coverage audit:')
print(reddit_coverage_audit.to_string(index=False))

reddit_coverage_audit

Raw source audit:
ticker           company_name        source_name                        selected_file  used_fallback  rows   date_min   date_max
  AAPL                  Apple              gdelt                  gdelt_data_aapl.csv          False  1078 2023-01-01 2025-12-31
  AAPL                  Apple      google_trends          google_trends_data_aapl.csv          False  1096 2023-01-01 2025-12-31
  AAPL                  Apple    reddit_comments  stock-reddit-comments-data_aapl.csv          False  1095 2023-01-01 2025-12-30
  AAPL                  Apple reddit_submissions           stock-reddit-data_aapl.csv          False  1095 2023-01-01 2025-12-30
  AAPL                  Apple              stock           stock-prices-data_aapl.csv          False   752 2023-01-03 2025-12-31
   AMD Advanced Micro Devices              gdelt                   gdelt_data_amd.csv          False  1078 2023-01-01 2025-12-31
   AMD Advanced Micro Devices      google_trends           google_trends_data_a

,ticker,company_name,subm_source_file,comm_source_file,subm_days,subm_days_with_posts,subm_post_coverage,subm_finbert_null_days,subm_finbert_zero_days,comm_days,comm_days_with_posts,comm_post_coverage,comm_finbert_null_days,comm_finbert_zero_days
0,AAPL,Apple,stock-reddit-data_aapl.csv,stock-reddit-comments-data_aapl.csv,1095,705,0.643836,390,0,1095,750,0.684932,345,0
1,AMD,Advanced Micro Devices,stock-reddit-data_amd.csv,stock-reddit-comments-data_amd.csv,1095,506,0.462100,589,0,1095,743,0.678539,352,0
2,AMZN,Amazon,stock-reddit-data_amzn.csv,stock-reddit-comments-data_amzn.csv,1095,664,0.606393,431,0,1095,750,0.684932,345,0
3,GOOGL,Alphabet,stock-reddit-data_googl.csv,stock-reddit-comments-data_googl.csv,1095,728,0.664840,367,0,1095,750,0.684932,345,0
4,META,Meta,stock-reddit-data_meta.csv,stock-reddit-comments-data_meta.csv,1095,568,0.518721,527,0,1095,748,0.683105,347,0
5,MSFT,Microsoft,stock-reddit-data_msft.csv,stock-reddit-comments-data_msft.csv,1095,661,0.603653,434,0,1095,750,0.684932,345,0
6,NFLX,Netflix,stock-reddit-data_nflx.csv,stock-reddit-comments-data_nflx.csv,1095,328,0.299543,767,0,1095,715,0.652968,380,0
7,NVDA,NVIDIA,stock-reddit-data_nvda.csv,stock-reddit-comments-data_nvda.csv,1095,691,0.631050,404,0,1095,750,0.684932,345,0
8,TSLA,Tesla,stock-reddit-data.csv,stock-reddit-comments-data.csv,1095,684,0.624658,411,0,1095,750,0.684932,345,0


In [7]:
panel = build_panel(TICKER_SPECS)
summary = build_panel_summary(panel)
metadata = build_metadata(source_audit, reddit_coverage_audit, panel)

source_audit.to_csv(OUTPUT_SOURCE_AUDIT_PATH, index=False)
reddit_coverage_audit.to_csv(OUTPUT_REDDIT_COVERAGE_PATH, index=False)
panel.to_csv(OUTPUT_DATASET_PATH, index=False)
summary.to_csv(OUTPUT_SUMMARY_PATH, index=False)

with open(OUTPUT_METADATA_PATH, 'w', encoding='utf-8') as handle:
    json.dump(metadata, handle, indent=2)

print(f'Saved source audit to {OUTPUT_SOURCE_AUDIT_PATH}')
print(f'Saved Reddit coverage audit to {OUTPUT_REDDIT_COVERAGE_PATH}')
print(f'Saved session-aligned panel to {OUTPUT_DATASET_PATH}')
print(f'Saved summary to {OUTPUT_SUMMARY_PATH}')
print(f'Saved metadata to {OUTPUT_METADATA_PATH}')

summary

Saved source audit to C:\Users\user\OneDrive\Documents\magisterka\praca magisterska\code\data\datasets\stock_panel_nine_tickers_session_aligned_source_audit.csv
Saved Reddit coverage audit to C:\Users\user\OneDrive\Documents\magisterka\praca magisterska\code\data\datasets\stock_panel_nine_tickers_session_aligned_reddit_coverage_audit.csv
Saved session-aligned panel to C:\Users\user\OneDrive\Documents\magisterka\praca magisterska\code\data\datasets\stock_panel_nine_tickers_session_aligned_raw.csv
Saved summary to C:\Users\user\OneDrive\Documents\magisterka\praca magisterska\code\data\datasets\stock_panel_nine_tickers_session_aligned_summary.csv
Saved metadata to C:\Users\user\OneDrive\Documents\magisterka\praca magisterska\code\data\datasets\stock_panel_nine_tickers_session_aligned_metadata.json


,metric,value
0,rows,6768
1,tickers,9
2,date_min,2023-01-03
3,date_max,2025-12-31
4,subm_finbert_null_rate,0.182181
5,comm_finbert_null_rate,0.009161
